# Batch Classify VAL Images

This notebook loads the trained CustomCNN model and the OCR text classifier, then sorts every image placed directly in `VAL` using the same multimodal ensemble strategy from `multimodal_ensemble.ipynb`.

Default behavior: move files from the top level of `VAL` into the predicted folder. The ensemble combines CNN and OCR probabilities, using the same threshold logic as the reference notebook.

In [19]:
import csv
import json
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms


def ensure_package(pkg, import_name=None):
    module_name = import_name or pkg.replace('-', '_')
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])


ensure_package('pytesseract')
ensure_package('pillow', 'PIL')
ensure_package('scikit-learn', 'sklearn')
ensure_package('joblib')
ensure_package('numpy')

import pytesseract
from joblib import load

In [20]:
# Paths, ensemble thresholds, and sorting configuration
PROJECT_ROOT = Path(r"c:\Users\ibf\Desktop\TFM\Nou projecte")
NOTEBOOK_ROOT = PROJECT_ROOT / "TFM"
MODEL_DIR = NOTEBOOK_ROOT / "Models"
CLASS_NAMES_PATH = MODEL_DIR / "class_names.json"
OCR_VECTORIZER_PATH = MODEL_DIR / "ocr_vectorizer.joblib"
OCR_CLASSIFIER_PATH = MODEL_DIR / "ocr_text_classifier.joblib"

MODEL_PATH = None  # Set a full .pth path here to force a specific checkpoint
MODEL_CANDIDATES = [
    "custom_cnn_from_scratch_DA_fairness.pth",
    "custom_cnn_from_scratch_30epochs.pth",
    "custom_cnn_from_scratch.pth",
]

INPUT_DIR = PROJECT_ROOT / "VAL"
OUTPUT_ROOT = INPUT_DIR
MOVE_FILES = True
OVERWRITE_EXISTING = False
SAVE_LOG = True
LOG_PATH = NOTEBOOK_ROOT / "val_multimodal_classification_results.csv"

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
IMAGE_SIZE = (224, 224)
BORDER_CROP = 120
COMBINED_THRESHOLD = 1.65
CNN_THRESHOLD = 0.85
OCR_THRESHOLD = 0.50

TESSERACT_CANDIDATES = [
    shutil.which("tesseract"),
    r"C:\Users\ibf\AppData\Local\Programs\Tesseract-OCR\tesseract.exe",
    r"C:\Program Files\Tesseract-OCR\tesseract.exe",
    r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Input folder: {INPUT_DIR}")
print(f"Output folder: {OUTPUT_ROOT}")
print(f"Move files: {MOVE_FILES}")
print(f"Combined threshold: {COMBINED_THRESHOLD:.2f}")
print(f"CNN threshold: {CNN_THRESHOLD:.2f}")
print(f"OCR threshold: {OCR_THRESHOLD:.2f}")

Using device: cpu
Input folder: c:\Users\ibf\Desktop\TFM\Nou projecte\VAL
Output folder: c:\Users\ibf\Desktop\TFM\Nou projecte\VAL
Move files: True
Combined threshold: 1.65
CNN threshold: 0.85
OCR threshold: 0.50


In [21]:
# Same architecture and preprocessing used during training
class CustomCNN(nn.Module):
    def __init__(self, num_classes=8):
        super(CustomCNN, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),
        )

        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.global_avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

def safe_mobile_border_crop(img):
    crop_margin = min(BORDER_CROP, max((img.height - 1) // 2, 0))
    return img.crop((0, crop_margin, img.width, img.height - crop_margin))

inference_transform = transforms.Compose([
    transforms.Lambda(safe_mobile_border_crop),
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [22]:
# Load CNN model, OCR artifacts, and configure Tesseract
if not CLASS_NAMES_PATH.exists():
    raise FileNotFoundError(f"class_names.json not found at: {CLASS_NAMES_PATH}")

if not OCR_VECTORIZER_PATH.exists():
    raise FileNotFoundError(f"OCR vectorizer not found at: {OCR_VECTORIZER_PATH}")

if not OCR_CLASSIFIER_PATH.exists():
    raise FileNotFoundError(f"OCR classifier not found at: {OCR_CLASSIFIER_PATH}")

if MODEL_PATH is None:
    resolved_model_path = None
    for model_name in MODEL_CANDIDATES:
        candidate = MODEL_DIR / model_name
        if candidate.exists():
            resolved_model_path = candidate
            break
    if resolved_model_path is None:
        raise FileNotFoundError(
            f"No model checkpoint found in {MODEL_DIR}. Checked: {MODEL_CANDIDATES}"
        )
else:
    resolved_model_path = Path(MODEL_PATH)
    if not resolved_model_path.exists():
        raise FileNotFoundError(f"Model file not found at: {resolved_model_path}")

with open(CLASS_NAMES_PATH, "r", encoding="utf-8") as f:
    class_names = json.load(f)

model = CustomCNN(num_classes=len(class_names)).to(device)
checkpoint = torch.load(resolved_model_path, map_location=device)
state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()

vectorizer = load(OCR_VECTORIZER_PATH)
ocr_clf = load(OCR_CLASSIFIER_PATH)
ocr_classes = list(ocr_clf.classes_)

resolved_tesseract_path = None
for candidate in TESSERACT_CANDIDATES:
    if candidate and Path(candidate).exists():
        resolved_tesseract_path = candidate
        break

if resolved_tesseract_path is not None:
    pytesseract.pytesseract.tesseract_cmd = resolved_tesseract_path
    print(f"Tesseract: {resolved_tesseract_path}")
else:
    print("Warning: Tesseract executable not found. OCR predictions will fail until Tesseract is installed.")

print(f"Loaded model: {resolved_model_path}")
print(f"Classes ({len(class_names)}): {class_names}")
print(f"OCR vectorizer: {OCR_VECTORIZER_PATH}")
print(f"OCR classifier: {OCR_CLASSIFIER_PATH}")

Tesseract: C:\Users\ibf\AppData\Local\Programs\Tesseract-OCR\tesseract.exe
Loaded model: c:\Users\ibf\Desktop\TFM\Nou projecte\TFM\Models\custom_cnn_from_scratch_DA_fairness.pth
Classes (8): ['Banner aplicación', 'Cierre aplicación', 'Error aplicativo', 'Error funcional', 'Error terminal', 'Indeterminado', 'Revisión circuito', 'Timeout']
OCR vectorizer: c:\Users\ibf\Desktop\TFM\Nou projecte\TFM\Models\ocr_vectorizer.joblib
OCR classifier: c:\Users\ibf\Desktop\TFM\Nou projecte\TFM\Models\ocr_text_classifier.joblib


In [9]:
def is_image_file(path):
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def unique_destination_path(destination_dir, file_name):
    candidate = destination_dir / file_name
    if OVERWRITE_EXISTING or not candidate.exists():
        return candidate

    stem = candidate.stem
    suffix = candidate.suffix
    index = 1
    while candidate.exists():
        candidate = destination_dir / f"{stem}_{index}{suffix}"
        index += 1
    return candidate


def safe_mobile_border_crop(img):
    crop_margin = min(BORDER_CROP, max((img.height - 1) // 2, 0))
    return img.crop((0, crop_margin, img.width, img.height - crop_margin))


inference_transform = transforms.Compose([
    transforms.Lambda(safe_mobile_border_crop),
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def extract_ocr_text(image_path):
    image = Image.open(image_path).convert("RGB")
    return pytesseract.image_to_string(image)


def predict_multimodal_image(image_path):
    image = Image.open(image_path).convert("RGB")
    input_tensor = inference_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        cnn_prob = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    text = extract_ocr_text(image_path)
    ocr_prob = ocr_clf.predict_proba(vectorizer.transform([text]))[0]

    predicted_label, confidence, reason = ensemble_predict_row(
        cnn_prob,
        ocr_prob,
        class_names,
        ocr_classes,
        combined_thresh=COMBINED_THRESHOLD,
        cnn_thresh=CNN_THRESHOLD,
        ocr_thresh=OCR_THRESHOLD,
    )

    cnn_idx = int(np.argmax(cnn_prob))
    ocr_idx = int(np.argmax(ocr_prob))

    return {
        "source_path": str(image_path),
        "predicted_class": predicted_label,
        "confidence": float(confidence),
        "reason": reason,
        "cnn_label": class_names[cnn_idx],
        "cnn_conf": float(cnn_prob[cnn_idx]),
        "ocr_label": ocr_classes[ocr_idx],
        "ocr_conf": float(ocr_prob[ocr_idx]),
        "text": text,
    }


def ensemble_predict_row(cnn_prob, ocr_prob, cnn_classes, ocr_classes, combined_thresh=1.65, cnn_thresh=0.80, ocr_thresh=0.50):
    ocr_aligned = np.zeros_like(cnn_prob)
    for i, lbl in enumerate(ocr_classes):
        if lbl in cnn_classes:
            j = cnn_classes.index(lbl)
            ocr_aligned[j] = ocr_prob[i]

    combined = cnn_prob + ocr_aligned
    idx = int(np.argmax(combined))
    combined_max = float(combined[idx])
    combined_label = cnn_classes[idx]

    cnn_idx = int(np.argmax(cnn_prob))
    cnn_label = cnn_classes[cnn_idx]
    cnn_conf = float(cnn_prob[cnn_idx])

    ocr_idx = int(np.argmax(ocr_prob))
    ocr_label = ocr_classes[ocr_idx]
    ocr_conf = float(ocr_prob[ocr_idx])

    if combined_max >= combined_thresh:
        return combined_label, combined_max, "combined_threshold"

    if cnn_conf >= cnn_thresh and ocr_conf >= ocr_thresh:
        return cnn_label, cnn_conf, "cnn_high_ocr_ok"

    if combined_max >= 1.00:
        return combined_label, combined_max, "combined_fallback"

    return "Indeterminado", combined_max, "indeterminado_low_combined"


def sort_images_in_folder(source_dir=INPUT_DIR, output_root=OUTPUT_ROOT):
    source_dir = Path(source_dir)
    output_root = Path(output_root)

    if not source_dir.exists():
        raise FileNotFoundError(f"Input folder not found: {source_dir}")

    image_paths = sorted(path for path in source_dir.iterdir() if is_image_file(path))

    if not image_paths:
        print(f"No image files found directly inside: {source_dir}")
        return [], Counter()

    results = []
    counts = Counter()

    for image_path in image_paths:
        prediction = predict_multimodal_image(image_path)
        destination_dir = output_root / prediction["predicted_class"]
        destination_dir.mkdir(parents=True, exist_ok=True)
        destination_path = unique_destination_path(destination_dir, image_path.name)

        if MOVE_FILES:
            shutil.move(str(image_path), str(destination_path))
        else:
            shutil.copy2(str(image_path), str(destination_path))

        row = {
            "source_path": prediction["source_path"],
            "predicted_class": prediction["predicted_class"],
            "confidence": prediction["confidence"],
            "reason": prediction["reason"],
            "cnn_label": prediction["cnn_label"],
            "cnn_conf": prediction["cnn_conf"],
            "ocr_label": prediction["ocr_label"],
            "ocr_conf": prediction["ocr_conf"],
            "destination_path": str(destination_path),
        }
        results.append(row)
        counts[prediction["predicted_class"]] += 1

        print(
            f"{image_path.name} -> CNN {prediction['cnn_conf']:.2%} '{prediction['cnn_label']}', "
            f"OCR {prediction['ocr_conf']:.2%} '{prediction['ocr_label']}', "
            f"Combined {prediction['confidence']:.2f}, Final: '{prediction['predicted_class']}'"
        )

    return results, counts

In [10]:
results, counts = sort_images_in_folder()

if results:
    print(f"\nSorted {len(results)} images.")
    print("Summary:")
    for folder_name, count in counts.most_common():
        print(f"  {folder_name}: {count}")

    if SAVE_LOG:
        with open(LOG_PATH, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=[
                    "source_path",
                    "predicted_class",
                    "confidence",
                    "reason",
                    "cnn_label",
                    "cnn_conf",
                    "ocr_label",
                    "ocr_conf",
                    "destination_path",
                ],
            )
            writer.writeheader()
            writer.writerows(results)
        print(f"\nSaved log to: {LOG_PATH}")
else:
    print("Nothing to process. Place the images you want to sort directly inside VAL and run this cell again.")

20260519011515_100134.png -> CNN 92.41% 'Revisión circuito', OCR 61.04% 'Revisión circuito', Combined 0.92, Final: 'Revisión circuito'
20260519013338_100187.png -> CNN 88.26% 'Revisión circuito', OCR 86.76% 'Revisión circuito', Combined 1.75, Final: 'Revisión circuito'
20260519014140_100126.png -> CNN 81.60% 'Revisión circuito', OCR 36.74% 'Revisión circuito', Combined 1.18, Final: 'Revisión circuito'
20260519021151_100133.png -> CNN 92.35% 'Revisión circuito', OCR 25.27% 'Revisión circuito', Combined 1.18, Final: 'Revisión circuito'
20260519024245_100126.png -> CNN 83.60% 'Revisión circuito', OCR 80.28% 'Revisión circuito', Combined 1.64, Final: 'Revisión circuito'
20260519030507_100048.png -> CNN 84.14% 'Revisión circuito', OCR 81.36% 'Revisión circuito', Combined 1.65, Final: 'Revisión circuito'
20260519032334_100126.png -> CNN 83.60% 'Revisión circuito', OCR 83.45% 'Revisión circuito', Combined 1.67, Final: 'Revisión circuito'
20260519034313_100105.png -> CNN 79.45% 'Error terminal

PermissionError: [WinError 32] El proceso no tiene acceso al archivo porque está siendo utilizado por otro proceso: 'C:\\Users\\ibf\\AppData\\Local\\Temp\\tess__rz12yul_input.PNG'